# OVD-Diagnose — Domain 1: Aerial (LAE-80C)

Frozen OVD detectors in Global / Oracle / Agnostic modes → (L, S, C) fingerprint with
detector-intrinsic `L_det`, bootstrap CIs, reliability diagrams, qualitative examples, and
the three synthetic controls (temperature / vocab random-vs-semantic / blur).

**Headless-ready:** Save Version → Save & Run All (Commit). GPU + Internet on.
This notebook carries the *validation* evidence for the protocol.

In [ ]:
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count())

In [ ]:
REPO = 'https://github.com/ShMazumder/YOLOBERT.git'
import os
if not os.path.isdir('/kaggle/working/YOLOBERT'):
    !git clone $REPO /kaggle/working/YOLOBERT
%cd /kaggle/working/YOLOBERT
!git pull -q || true

In [ ]:
!pip install -q ultralytics transformers pycocotools pyyaml sentence-transformers
!python tools/ovd_diagnose.py   # metric self-test

## Data — LAE-80C (download + recursive flatten, verified against annotation count)

In [ ]:
import os, zipfile, shutil, json
BASE = '/kaggle/working/YOLOBERT/data/aerial'
ANN_DIR, IMG_DIR = f'{BASE}/annotations', f'{BASE}/images'
os.makedirs(ANN_DIR, exist_ok=True); os.makedirs(IMG_DIR, exist_ok=True)
ann = f'{ANN_DIR}/instances_val.json'
HF = 'https://huggingface.co/datasets/jaychempan/LAE-1M/resolve/main/LAE-80C'
if not os.path.exists(ann):
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.json?download=true" -O {ann}
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark_categories.json?download=true" -O {ANN_DIR}/categories.json
    !wget -q --no-check-certificate "{HF}/LAE-80C-benchmark.txt?download=true" -O {ANN_DIR}/classes.txt
def count_jpgs():
    return sum(1 for _, _, fs in os.walk(IMG_DIR) for f in fs if f.lower().endswith('.jpg'))
n_expected = len(json.load(open(ann))['images'])
if count_jpgs() < n_expected:
    zp = '/kaggle/working/images.zip'
    if not os.path.exists(zp) or os.path.getsize(zp) < 5e8:
        !wget -q --no-check-certificate "{HF}/images.zip?download=true" -O {zp}
    with zipfile.ZipFile(zp) as z: z.extractall(IMG_DIR)
    if os.path.exists(zp): os.remove(zp)
for root, _, files in os.walk(IMG_DIR):
    if root == IMG_DIR: continue
    for f in files:
        if f.lower().endswith('.jpg'):
            dst = os.path.join(IMG_DIR, f)
            if not os.path.exists(dst): shutil.move(os.path.join(root, f), dst)
for root, dirs, files in os.walk(IMG_DIR, topdown=False):
    if root != IMG_DIR and not os.listdir(root): os.rmdir(root)
root_jpgs = len([f for f in os.listdir(IMG_DIR) if f.lower().endswith('.jpg')])
print(f'images: {root_jpgs} | expected: {n_expected} | complete: {root_jpgs >= n_expected}')

## Run all models (`LIMIT=0` full split; `200` quick). `--bootstrap 1000` adds IoA-F1 CIs + n_images.

In [ ]:
LIMIT = 0
limit_arg = f'--limit {LIMIT}' if LIMIT else ''
!python tools/run_all.py \
  --ann  data/aerial/annotations/instances_val.json \
  --imgs data/aerial/images \
  --out  runs/diag/aerial \
  --device cuda:0 {limit_arg} --bootstrap 1000 \
  --models "yoloworld:yolov8s-world.pt,owlv2:google/owlv2-base-patch16-ensemble,groundingdino:IDEA-Research/grounding-dino-tiny" \
  --sam_weights mobile_sam.pt

In [ ]:
# Fingerprint table: L (SAM), L_det (detector-intrinsic), S, C, IoA-F1 + 95% CI, n_images
import pandas as pd
print(pd.read_csv('runs/diag/aerial/fingerprints.csv').to_string(index=False))

## Calibration — reliability diagrams

In [ ]:
!python tools/plot_reliability.py \
  --ann data/aerial/annotations/instances_val.json \
  --models yoloworld:runs/diag/aerial/yoloworld/results_global.json \
           owlv2:runs/diag/aerial/owlv2/results_global.json \
           groundingdino:runs/diag/aerial/groundingdino/results_global.json \
  --out paper/figures/reliability_aerial

## Qualitative — semantic confusion (localized but wrong class)

In [ ]:
!python tools/qualitative.py \
  --ann data/aerial/annotations/instances_val.json --imgs data/aerial/images \
  --results runs/diag/aerial/owlv2/results_global.json \
  --mode localized_wrong --n 6 --out paper/figures/qual_aerial.png
from IPython.display import Image
Image('paper/figures/qual_aerial.png')

## Validation — synthetic controls
Each perturbs one factor; only its axis should move. This is the protocol's construct validity.

In [ ]:
# 1) Temperature (post-hoc, no GPU): C_ece must vary while AP (hence S) stays flat.
!python tools/synthetic_controls.py --control temperature \
  --ann data/aerial/annotations/instances_val.json \
  --results runs/diag/aerial/owlv2/results_global.json \
  --out runs/controls/aerial_owlv2_temperature.csv

In [ ]:
# 2) Vocabulary: RANDOM vs SEMANTIC distractors.
#    semantic >= random at each count => S measures semantic confusion, not vocab size.
for mode in ['random', 'semantic']:
    !python tools/synthetic_controls.py --control vocab --distractor {mode} \
      --ann data/aerial/annotations/instances_val.json --imgs data/aerial/images \
      --model owlv2 --weights google/owlv2-base-patch16-ensemble \
      --out runs/controls/aerial_vocab_{mode}.csv --limit 300 --device cuda:0

In [ ]:
# 3) Blur: L must rise with input degradation (text-independent).
!python tools/synthetic_controls.py --control blur \
  --ann data/aerial/annotations/instances_val.json --imgs data/aerial/images \
  --sam_weights mobile_sam.pt --out runs/controls/aerial_blur.csv --limit 200 --device cuda:0

In [ ]:
import pandas as pd, os
for name in ['aerial_owlv2_temperature', 'aerial_vocab_random', 'aerial_vocab_semantic', 'aerial_blur']:
    p = f'runs/controls/{name}.csv'
    if os.path.exists(p):
        print(f'== {name} =='); print(pd.read_csv(p).to_string(index=False), '\n')

**Pass criteria:** temperature → `C_ece` varies, `AP` flat. vocab → `S` rises with distractor
count **and semantic ≥ random**. blur → `L` rises. Plus `L_det` ≈ SAM-based `L`, and bootstrap
CIs on IoA-F1. Together these answer the reviewer concerns: axis identifiability, S validity,
calibration relevance, and uncertainty.

Next: run `ovd_diagnose_medical.ipynb` for the cross-domain contrast.